# Доверительные интервалы


**Исполнители (ФИО):** Липунцов Антон Юрьевич

---

Здравствуйте! В этом практикуме мы перейдем от задачи точечного оценивания к задаче доверительного оценивания

Хотя точечные оценки при достаточном размере выборки хорошо приближают истинный параметр и позволяют построить функцию распределения, в общем случае невозможно понять насколько точно нам удалось оценить истинное значение параметра и сильно ли мы ошиблись. Для решения этой проблемы мы можем построить доверительный интервал

Точным доверительным интервалом с уровнем доверия $1-\alpha$ (обычно берут 95%) мы называем пару статистик $\hat{\theta_1}, \hat{\theta_2}$, таких, что при всех $\theta$ выполнено равенство $P_{\theta}(\theta \in (\hat{\theta_1}, \hat{\theta_2})) = 1 - \alpha$.

Чтобы его построить нужно найти статистику и функцию от нее и $\theta$ с известной функцией распределения, определить интервал, в котором функция лежит с заданным уровнем доверия и затем обратить ее по $\theta$.



## Задача 1

Сгенерируйте выборку из распределения $X\sim R[\theta;\theta+1]$, значение параметра $\theta$ выберите сами

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, uniform

theta_true = 2.0
n = 30
sample = uniform.rvs(loc=theta_true, scale=1.0, size=n)

Постройте точный доверительный интервал для $\theta$ на основе $X_{(n)}$

In [28]:
alpha = 0.05
X_n = np.max(sample)

a = (alpha/2)**(1/n)
b = (1 - alpha/2)**(1/n)

ci_lower = X_n - b
ci_upper = X_n - a

print(f"Точный доверительный интервал для θ (уровень {1-alpha}): [{ci_lower}, {ci_upper}]")
print(f"Длина интервала: {ci_upper - ci_lower}")

Точный доверительный интервал для θ (уровень 0.95): [1.9963170135816022, 2.0204408396046007]
Длина интервала: 0.024123826022998562


Проверьте численно, что уровень доверия построенного интервала не зависит от размера выборки и равен $1-\alpha$

In [ ]:
alpha = 0.05
n_vals = [10, 30, 50, 100]
n_sim = 1000
coverage = []

for n in n_vals:
    covered = 0
    for _ in range(n_sim):
        theta = 2.0
        samp = uniform.rvs(loc=theta, scale=1.0, size=n)
        Xn = np.max(samp)
        a = (alpha/2)**(1/n)
        b = (1 - alpha/2)**(1/n)
        lower = Xn - b
        upper = Xn - a
        if lower <= theta <= upper:
            covered += 1
    coverage.append(covered / n_sim)

for n, cov in zip(n_vals, coverage):
    print(f"n = {n}, уровень доверия = {cov}")

n = 10, уровень доверия = 0.953
n = 30, уровень доверия = 0.954
n = 50, уровень доверия = 0.946
n = 100, уровень доверия = 0.951


**Вопрос:** Какая получилась длина построенного интервала? Является ли она минимальной? Зависит ли уровень доверия интервала от размера выборки? Обоснуйте ответ

$b-a=(1-\frac{α}{2})^\frac{1}{n}-(\frac{α}{2})^\frac{1}{n}$  
Уровень доверия всегда один и тот же, но длина уменьшается при увеличении размера выборки.

## Задача 2

Однако точные интервалы строить проблематично, потому на практике чаще строят асимптотические доверительные интервалы: $P_{\theta}(\theta \in (\hat{\theta_1}, \hat{\theta_2})) \rightarrow 1 - \alpha$ при $n \rightarrow \infty$. Для асимптотически нормальной оценки $\hat{\theta}$, исходя из ее определения, можно построить асимптотический доверительный интервал вида $(\hat{\theta} - \frac{z_{1-\alpha/2}\sigma(\hat{\theta})}{\sqrt{n}}; \hat{\theta} + \frac{z_{1-\alpha/2}\sigma(\hat{\theta})}{\sqrt{n}})$

Постройте асимптотический доверительный интервал с уровнем доверия 95% для параметра $\theta$ из задачи 1

In [ ]:
alpha = 0.05
n = len(sample)
theta_hat = np.mean(sample) - 0.5
std_theta = 1 / np.sqrt(12 * n)
z = norm.ppf(1 - alpha/2)

ci_lower_as = theta_hat - z * std_theta
ci_upper_as = theta_hat + z * std_theta

print(f"Асимптотический интервал: [{ci_lower_as:.4f}, {ci_upper_as:.4f}]")
print(f"Длина: {ci_upper_as - ci_lower_as:.4f}")

Асимптотический интервал: [1.9110, 2.1176]
Длина: 0.2066


Посчитайте эмпирическую доверительную вероятность интервала, для этого сгенерируйте 1000 выборок и постройте для каждой доверительный интервал, затем посчитайте долю тех, в которые попало истинное значение параметра

In [ ]:
n_sim = 1000
n = 30
alpha = 0.05
z = norm.ppf(1 - alpha/2)
true_theta = 2.0

covered = 0
for _ in range(n_sim):
    samp = uniform.rvs(loc=true_theta, scale=1.0, size=n)
    theta_hat = np.mean(samp) - 0.5
    std_theta = 1 / np.sqrt(12 * n)
    lower = theta_hat - z * std_theta
    upper = theta_hat + z * std_theta
    if lower <= true_theta <= upper:
        covered += 1

print(f"Эмпирическая вероятность покрытия (n={n}): {covered/n_sim}")

Эмпирическая вероятность покрытия (n=30): 0.956


Постройте интервалы для разного размера выборки, сравните их эмпирическую доверительную вероятность

In [ ]:
n_vals = [10, 30, 50, 100]
coverages = []
lengths = []

for n in n_vals:
    covered = 0
    length_sum = 0
    for _ in range(n_sim):
        samp = uniform.rvs(loc=true_theta, scale=1.0, size=n)
        theta_hat = np.mean(samp) - 0.5
        std_theta = 1 / np.sqrt(12 * n)
        lower = theta_hat - z * std_theta
        upper = theta_hat + z * std_theta
        length_sum += (upper - lower)
        if lower <= true_theta <= upper:
            covered += 1
    coverages.append(covered / n_sim)
    lengths.append(length_sum / n_sim)

for n, cov, L in zip(n_vals, coverages, lengths):
    print(f"n = {n}, покрытие = {cov}, средняя длина = {L}")

n = 10, покрытие = 0.954, средняя длина = 0.3578388287434274
n = 30, покрытие = 0.945, средняя длина = 0.20659834410151645
n = 50, покрытие = 0.962, средняя длина = 0.16003038921184115
n = 100, покрытие = 0.961, средняя длина = 0.11315857340761683


**Вопрос:** Как зависит длина асимптотически доверительного интервала и эмпирическая доверительная вероятность в зависимости от размера  выборки? Как отличается длина и уровень доверия в сравнении с точным интервалом? Предположите, почему?

При увеличении объёма выборки длина убывает как 1/n, эмпирическая доверительная вероятность стремится к 1-a. При небольших выборках точный интервал обеспечивает заявленный уровень лучше,
асимптотический может как недокрывать, так и перекрывать, при больших размерах длины и уровень доверия близки.

## Задача 3

Другой подход предлагает использовать бутстреп для построения интервалов. Его две основные идеи:

1. Если мы знаем функцию распределения какой-то выборки, то с помощью метода Монте-Карло мы
можем приближенно найти моменты функций от наших величин, генерируя выборки из нашего
распределения.
2. Если у нас есть "хорошая оценка" $\hat{\theta}$ неизвестного параметра  $\theta$, а семейство распределений непрерывно, то $F_{\hat{\theta}}$ близко к $F_{\theta}$.

Рассмотрим оценку $\hat{\theta}$ для $\theta$. Будем брать выборки из $F_{\hat{\theta}}$ и строить на основе них оценки $\hat{\theta_1}, ..., \hat{\theta_m}$. Далее есть несколько простых вариантов построения интервала:

Normal - для асимптотически нормальных оценок можно построить интервал вида $(\hat{\theta} - z_{1-\alpha/2}\hat{\sigma}; \hat{\theta} + z_{1-\alpha/2}\hat{\sigma})$, где $\hat{\sigma}$ это бутстреп оценка для дисперсии $\hat{\theta}$

Percentile - в качестве интервала предлагается взять интервал $(\hat{\theta_{(i)}};\hat{\theta_{(j)}})$ таких, что между ними лежит $(1-\alpha)$% всех $\hat{\theta_i}$

Постройте при помощи бутстреп Normal, Percentile доверительные интервалы с уровнем доверия 95% для параметра $\theta$ из задачи 1, сравните их с интервалами из задач 1, 2

In [ ]:
def bootstrap_ci(sample, B=1000, alpha=0.05):
    n = len(sample)
    theta_hat = np.mean(sample) - 0.5
    boot_estimates = []
    for _ in range(B):
        boot_samp = np.random.choice(sample, size=n, replace=True)
        boot_estimates.append(np.mean(boot_samp) - 0.5)
    boot_estimates = np.array(boot_estimates)

    std_boot = np.std(boot_estimates, ddof=1)
    z = norm.ppf(1 - alpha/2)
    norm_lower = theta_hat - z * std_boot
    norm_upper = theta_hat + z * std_boot

    perc_lower = np.percentile(boot_estimates, 100 * alpha/2)
    perc_upper = np.percentile(boot_estimates, 100 * (1 - alpha/2))

    return (norm_lower, norm_upper), (perc_lower, perc_upper)

norm_ci, perc_ci = bootstrap_ci(sample, B=1000)
print("Bootstrap Normal CI:", norm_ci)
print("Bootstrap Percentile CI:", perc_ci)

Bootstrap Normal CI: (np.float64(1.9046175499768023), np.float64(2.1240276409394965))
Bootstrap Percentile CI: (np.float64(1.906014063708061), np.float64(2.121986299011203))


In [ ]:
print(f"Точный: [{ci_lower}, {ci_upper}], {ci_upper-ci_lower}")
print(f"Асимптотический: [{ci_lower_as}, {ci_upper_as}], {ci_upper_as-ci_lower_as}")
print(f"Bootstrap Normal: [{norm_ci[0]}, {norm_ci[1]}], {norm_ci[1] - norm_ci[0]}")
print(f"Bootstrap Percentile: [{perc_ci[0]}, {perc_ci[1]}], {perc_ci[1] - perc_ci[0]}")

Точный: [1.9969918133651228, 2.11185155066052], 0.11485973729539722
Асимптотический: [1.911023423407389, 2.1176217675089095], 0.20659834410152045
Bootstrap Normal: [1.9046175499768023, 2.1240276409394965], 0.21941009096269415
Bootstrap Percentile: [1.906014063708061, 2.121986299011203], 0.21597223530314213


**Вопрос:** Чем отличаются построенные интервалы? Какой по вашему мнению оказался лучше? Ответ обоснуйте

Точный лучше, но требует знания распределения в явном виде. Среди бутстреп лучше Percentile, поскольку распределение оценки параметра θ не является симметричным.

Посчитайте эмпирическую доверительную вероятность интервалов, для этого сгенерируйте 1000 выборок и постройте для каждой доверительный интервал, затем посчитайте долю тех, в которые попало истинное значение параметра

In [ ]:
def empirical_coverage_bootstrap(n, true_theta=2.0, n_sim=1000, B=200, alpha=0.05):
    covered_norm = 0
    covered_perc = 0
    for _ in range(n_sim):
        samp = uniform.rvs(loc=true_theta, scale=1.0, size=n)
        norm_ci, perc_ci = bootstrap_ci(samp, B=B, alpha=alpha)
        if norm_ci[0] <= true_theta <= norm_ci[1]:
            covered_norm += 1
        if perc_ci[0] <= true_theta <= perc_ci[1]:
            covered_perc += 1
    return covered_norm/n_sim, covered_perc/n_sim

n_vals = [50, 100, 300]
for n in n_vals:
    cov_norm, cov_perc = empirical_coverage_bootstrap(n, n_sim=300, B=200)
    print(f"n={n}: Normal = {cov_norm}, Percentile = {cov_perc}")

n=50: Normal = 0.9266666666666666, Percentile = 0.92
n=100: Normal = 0.9566666666666667, Percentile = 0.95
n=300: Normal = 0.94, Percentile = 0.9333333333333333


Постройте интервалы для разного размера выборки, сравните их эмпирическую доверительную вероятность. Добавьте к сравнению точный и асимптотический доверительные интервалы

In [ ]:
n_sim = 500

for n in n_vals:
    cov_exact = 0
    cov_asymp = 0
    cov_norm, cov_perc = empirical_coverage_bootstrap(n, n_sim=300, B=200)
    for _ in range(n_sim):
        samp = uniform.rvs(loc=true_theta, scale=1.0, size=n)
        Xn = np.max(samp)
        a = (alpha/2)**(1/n)
        b = (1 - alpha/2)**(1/n)
        lower_exact = Xn - b
        upper_exact = Xn - a
        if lower_exact <= true_theta <= upper_exact:
            cov_exact += 1
        theta_hat = np.mean(samp) - 0.5
        std_theta = 1 / np.sqrt(12 * n)
        lower_as = theta_hat - z * std_theta
        upper_as = theta_hat + z * std_theta
        if lower_as <= true_theta <= upper_as:
            cov_asymp += 1
    print(f"n={n:2d}: Точный = {cov_exact/n_sim:.3f}, Асимптот. = {cov_asymp/n_sim:.3f}, Normal = {cov_norm}, Percentile = {cov_perc}")

n=50: Точный = 0.936, Асимптот. = 0.940, Normal = 0.9366666666666666, Percentile = 0.92
n=100: Точный = 0.958, Асимптот. = 0.954, Normal = 0.9366666666666666, Percentile = 0.9266666666666666
n=300: Точный = 0.948, Асимптот. = 0.948, Normal = 0.9333333333333333, Percentile = 0.9166666666666666


**Вопрос:** Какой из методов показал лучшую эмперическую доверительную вероятность бутстреп, точный или асимптотический? Предположите, почему?

## Задача 4

В файле *unif.txt* приведены 150 измерений датчика спутника в диапазоне $[0; a]$

Загрузите датасет

In [ ]:
data = np.loadtxt("unif.txt")

Постройте доверительный интервал для измеряемой величины понравившимся вам способом. Локализуйте $a$ так,чтобы вероятность вашей ошибки была не более $1\%$

In [ ]:
alpha = 0.01
n = len(data)
M = np.max(data)

lower_a = M / ((1 - alpha/2)**(1/n))
upper_a = M / ((alpha/2)**(1/n))

print(f"Доверительный интервал для a: [{lower_a}, {upper_a}]")
print(f"Длина интервала: {upper_a - lower_a}")

Доверительный интервал для a: [4.965851940830986, 5.144219056579003]
Длина интервала: 0.1783671157480171


**Вопрос:** Объясните полученный результат и обоснуйте выбор метода

*Your answer here*